# BE Sphere Density Analysis

This notebook analyzes the density distribution from the Phase 1 BE Sphere relaxation simulation.

**Configuration:**
- Mass: 40 M☉ isothermal Bonnor-Ebert sphere
- ξ_s = 6.0 (stable, below critical ξ_crit = 6.451)
- Central density: n_center = 1800 cm⁻³
- Edge density: n_edge = 162 cm⁻³
- Temperature: T = 7 K

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Configure matplotlib for better plots
plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'numpy'

## 2. Load Configuration File

In [ ]:
# Define paths
BASE_DIR = Path("/Users/guo-opt-p148/sph-code-simulation/simulations/astrophysics/imbh_cloud")
CONFIG_FILE = BASE_DIR / "config/presets/phase1_relaxation.json"
RESULTS_DIR = BASE_DIR / "results/phase1_relaxation"

# Load configuration
with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)

# Display key parameters
print("=" * 50)
print("BE Sphere Configuration")
print("=" * 50)
print(f"Cloud Mass:      {config['M_cloud']} M☉")
print(f"Temperature:     {config['T_cloud']} K")
print(f"Central n:       {config['n_center']} cm⁻³")
print(f"Edge n:          {config['n_edge']} cm⁻³")
print(f"ξ_s:             {config['xi_s']}")
print(f"Radius:          {config['R_cloud']} pc")
print(f"Stability:       {config['stability_analysis']['status']}")
print(f"Density Contrast: {config['stability_analysis']['density_contrast']}")

## 3. Load Simulation Output Data

The simulation has already been run. Load the final snapshot from the results directory.

In [ ]:
# List available snapshots
snapshots = sorted(RESULTS_DIR.glob("snapshot_*.csv"))
print(f"Found {len(snapshots)} snapshots:")
for s in snapshots:
    print(f"  - {s.name}")

# Load the final snapshot
if snapshots:
    final_snapshot = snapshots[-1]
    print(f"\nLoading final snapshot: {final_snapshot.name}")
    
    # Read CSV, skipping comment lines
    df = pd.read_csv(final_snapshot, comment='#')
    
    # Filter out ghost particles
    df_real = df[df['is_ghost'] == 0].copy()
    
    print(f"\nData loaded:")
    print(f"  Total particles: {len(df)}")
    print(f"  Real particles:  {len(df_real)}")
    print(f"  Ghost particles: {len(df) - len(df_real)}")
    print(f"\nColumns: {list(df.columns)}")
else:
    print("No snapshots found!")

## 4. Plot Radial Density Profile

In [ ]:
# Calculate radius from center
df_real['r'] = np.sqrt(df_real['pos_x']**2 + df_real['pos_y']**2 + df_real['pos_z']**2)

# Create figure with 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Density vs radius scatter plot (colored by neighbor count)
ax1 = axes[0, 0]
scatter = ax1.scatter(df_real['r'], df_real['dens'], 
                      c=df_real['neighbor'], cmap='viridis', 
                      alpha=0.5, s=10)
ax1.set_xlabel('Radius [code units]', fontsize=12)
ax1.set_ylabel('Density [code units]', fontsize=12)
ax1.set_title('Density vs Radius (colored by neighbor count)', fontsize=14)
ax1.set_yscale('log')
plt.colorbar(scatter, ax=ax1, label='Neighbors')

# 2. Radially averaged density profile
ax2 = axes[0, 1]
n_bins = 50
r_bins = np.linspace(0, df_real['r'].max(), n_bins + 1)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])

density_mean = []
density_std = []
for i in range(n_bins):
    mask = (df_real['r'] >= r_bins[i]) & (df_real['r'] < r_bins[i+1])
    if mask.sum() > 0:
        density_mean.append(df_real.loc[mask, 'dens'].mean())
        density_std.append(df_real.loc[mask, 'dens'].std())
    else:
        density_mean.append(np.nan)
        density_std.append(np.nan)

density_mean = np.array(density_mean)
density_std = np.array(density_std)

ax2.plot(r_centers, density_mean, 'b-', linewidth=2, label='Mean density')
ax2.fill_between(r_centers, 
                 density_mean - density_std, 
                 density_mean + density_std,
                 alpha=0.3, label='±1σ')
ax2.set_xlabel('Radius [code units]', fontsize=12)
ax2.set_ylabel('Density [code units]', fontsize=12)
ax2.set_title('Radially Averaged Density Profile', fontsize=14)
ax2.set_yscale('log')
ax2.legend()

# 3. 2D slice (XY plane, near z=0)
ax3 = axes[1, 0]
z_slice_width = df_real['pos_z'].std() * 0.3
slice_mask = np.abs(df_real['pos_z']) < z_slice_width
df_slice = df_real[slice_mask]

scatter3 = ax3.scatter(df_slice['pos_x'], df_slice['pos_y'], 
                       c=np.log10(df_slice['dens']), cmap='hot', 
                       alpha=0.7, s=15)
ax3.set_xlabel('X [code units]', fontsize=12)
ax3.set_ylabel('Y [code units]', fontsize=12)
ax3.set_title(f'XY Slice (|z| < {z_slice_width:.3f})', fontsize=14)
ax3.set_aspect('equal')
plt.colorbar(scatter3, ax=ax3, label='log₁₀(Density)')

# 4. Density histogram
ax4 = axes[1, 1]
ax4.hist(df_real['dens'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax4.set_xlabel('Density [code units]', fontsize=12)
ax4.set_ylabel('Count', fontsize=12)
ax4.set_title('Density Distribution', fontsize=14)
ax4.axvline(df_real['dens'].mean(), color='r', linestyle='--', 
            label=f'Mean: {df_real["dens"].mean():.2f}')
ax4.axvline(df_real['dens'].median(), color='g', linestyle='--',
            label=f'Median: {df_real["dens"].median():.2f}')
ax4.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'density_profile.png', dpi=150, bbox_inches='tight')
print(f"Saved plot to: {RESULTS_DIR / 'density_profile.png'}")
plt.show()

## 5. Density Statistics

In [ ]:
# Print density statistics
print("Density Statistics for Final Snapshot")
print("=" * 50)
print(f"Number of particles: {len(df)}")
print(f"Mean density: {df['dens'].mean():.6e}")
print(f"Median density: {df['dens'].median():.6e}")
print(f"Min density: {df['dens'].min():.6e}")
print(f"Max density: {df['dens'].max():.6e}")
print(f"Std deviation: {df['dens'].std():.6e}")
print(f"Central density (r < 0.1 R_cloud): {df[df['radius'] < 0.1 * df['radius'].max()]['dens'].mean():.6e}")
print(f"Edge density (r > 0.8 R_cloud): {df[df['radius'] > 0.8 * df['radius'].max()]['dens'].mean():.6e}")
print(f"\nDensity contrast (center/edge): {df[df['radius'] < 0.1 * df['radius'].max()]['dens'].mean() / df[df['radius'] > 0.8 * df['radius'].max()]['dens'].mean():.2f}")

# Save the plot
output_path = results_dir / "density_profile.png"
fig.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Plot saved to: {output_path}")